# MediaVault · серверный импорт в Google Drive

Видео скачивается **на сервер Google Colab**, собирается в MP4, шифруется AES-GCM
и загружается в ваш Drive. Телефон передаёт небольшое задание; видеотрафик при
импорте не идёт через телефон, GitHub Pages или Cloudflare Worker.

1. На сайте MediaVault откройте свою библиотеку → **Импорт**. Укажите публичные
   ссылки, коллекцию и серии, нажмите **Подготовить серверный импорт → Скопировать задание**.
2. Здесь выберите **Среда выполнения → Выполнить всё**. Достаточно бесплатного CPU;
   GPU, Pro, банковская карта и Cloudflare не требуются.
3. Вставьте скопированное задание в появившееся поле. Затем войдите в аккаунт
   владельца и введите фразу библиотеки в скрытом поле `getpass`.
4. После завершения нажмите **Обновить библиотеку** на сайте.

Используется собственный разбор публичного HTML и открытые адаптеры плееров
`anime-dl-core`. API аниме-сайтов, DRM и платный доступ не используются.
Поддерживаются найденные публичные MP4/HLS и плееры Kodik, Aniboom, Sibnet,
Animedia. Если сайт отдаёт только проверку браузера или скрывает список серий,
понадобятся отдельные публичные ссылки; поддержка всех зеркал не гарантируется.

Бесплатные сессии Colab имеют переменную длительность и могут отключаться.
После каждой завершённой серии результат сохраняется в Drive. Повторный запуск
того же задания пропустит уже записанные серии. Временные файлы занимают место
только для текущей серии и удаляются после неё. Пока идёт импорт, не изменяйте
библиотеку с другого устройства. В Drive должно хватать свободного места.

**Не вставляйте пароль Google, Client Secret, токены или секретную фразу в JSON
задания или ячейки кода.** Фраза вводится только в скрытом поле при запуске и
не записывается в блокнот. Не делитесь работающей сессией и её выводом.

In [ ]:
# 1. Подготовка среды. Код загружается и проверяется до входа в Google.
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, urllib.request

SOURCE_HASHES = {'server/source_parser.py': '4ebd5cb336d960ccbfcf7bc23b9d0d7c92dd8326f040e655fe9e22ceca3d1f09', 'server/media_download.py': '665d5c6f46dc013dec616b15a65140c99fd03964f85a820c555014c8c7e8f8e9', 'server/run_import.py': '92125208479c3325a6ebb7ab0e43e9bb60306734661cef7783069f1db097f956', 'server/vault_drive.py': '46f0d760c335c276f34d8b705f8a8d54f61d98adc63343a7b905a4377f74c9df', 'server/player_catalog.py': '90a909760d591f4c368b0da801eba277a30c19fab458e261df1ddfda1fb34c18', 'server/player_compat.py': 'bd18fd66037d7bc9e2f30eaa34553d320f7477e07758b1a05c4e5da3766df3bc', 'server/browser_player.py': '6cbf9710385bf5b5a92e199db6eac4a6a6f43793268835530df7af3a5a8285f5', 'server/public_page_data.py': '9c94065d91ee3f418b19f9de248abacf64dda79fb2414879d15ea72aa9998fa0', 'server/requirements.txt': 'b73183846196beb769b03b61c427bf549c9b7e1051523d34c36aa2a35cba9953', 'server/title_extras.py': '679bb5169223e64051163ceb14bb533ecfa7363c41dbfd6a8804fc0b02cfd743', 'server/manga_sources.py': '2ff5ee8a979ed379ae2e380361dc186ff0416ae3328dc645782bfc6a2663056e', 'server/manga_novelcool.py': '744b7c5262b9b8169e1f00cc929f84aca6acdbdcd8d87eedd03ac02556e3dd20', 'server/novelcool_client.py': 'ba2faefe7ea37312954962f06fc9d84a283bda96d827c6bb8a16296c388bfd87', 'server/manga_raw_adapters.py': '9c3c012b62226861384beccdc0f11b0201c0a2aae14e501e54889984629a6152', 'server/manga_webtoons.py': 'e800693d1c28ea7fedf017dde3c2a9c981cb45a50dc73fa3608c58c85647d743', 'server/manga_catalog_adapters.py': 'b493a93bdee8296c063e1f4d6005cfe397a71dfa182459d4fe9a342ef340a1f0', 'server/manga_download.py': '91a016c946f3252d3d9219a6cf60f1198d78318f8817c6e50eed16272bc8fdf0', 'server/title_identity.py': '218d0761ac0536ff58f758fcd0e31bda578ddb7de4cb2b4bc54e3e78f2d95c10', 'server/suwayomi_bridge.py': '726fd84dd70d443abc95dbbdc17b2bb2e8a601264f2ae8d951943453669b7d9f', 'server/manga_hmanga.py': '6f2d4e717fe535b1c6dfefe7a0f6190460ed6c58867db888f520894ba97c0beb', 'server/manga_source_tags.py': '5793dce9272209eb4a63b93afe9ad503c4d6c238eb373061e64af0c89c990144', 'server/manga_source_coverage.py': 'e7b7d509517d78905ed580bcb35e226e72c0ec00e291fb710fdf7708b87a420b', 'server/manga_catalog_index.py': '09290a265f3527f3d6046de2926cf0d329c0737815c2df6570ba0b4e373ed403', 'server/manga_tag_policy.py': '96b7a90640c0d1e23628bdd3994cf934d112268abf25cc84285ebb4120c7793f', 'server/manga_tag_deploy.py': '273f4b2e6b9b5fabbd9bd55a844733f0acd776b201fb2493d43461e2f941217c', 'server/manga_catalog_metadata.py': '73ffe6274e77812395244a4e8a49e482e350486031554619cc556cb76d85ae36', 'server/content_catalog.py': 'ee47cef3913354a160c96ce4b15397c1ff7a37faecb3318f88b643c98f0be60d', 'server/content_management.py': '1af2758ab884ad28791d4749a7c00496c359e885ed08f49c54e6371385200ee3', 'server/content_sources.py': 'c54f4fe2926c957d334de49fa76a785d3fce3b2f0cbba66312a077f870315b8a', 'server/content_metadata.py': '68a6e2202e0742a672ff82598283b2b856537bbc8b737b77a35fcb1028bb23ae', 'server/content_source_routes.py': 'fe39199aca6c20e97c1a38817681e75cc6a2cdc9e7378a85a1c64656eb8518a3', 'server/content_vocabulary.py': 'f4cc84ea65988bdea39c1d8878da02cb239033327a4dfbd0fa28d1ca22091425', 'server/catalogue_sources.py': '14f83d1a855d93de9e00a3910eecce2b3235eb31bb77b2048e1e1fd4e7f2a353', 'server/catalogue_indexer.py': 'ceaf42ead05cf70f0f50c73fbbf2a1dff27ed651525072a427f2c485aa8f833d'}
CONFIG_FILES = ['server/manga-tag-policy.json']
BASE = "https://raw.githubusercontent.com/Onikri/MediaVault/main/"
RUNTIME = Path("/content/mediavault-server")
RUNTIME.mkdir(parents=True, exist_ok=True)
for relative, expected in SOURCE_HASHES.items():
    with urllib.request.urlopen(BASE + relative, timeout=45) as response:
        data = response.read(2 * 1024 * 1024)
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError("Исходный код обновился. Откройте свежий блокнот по ссылке с сайта.")
    (RUNTIME / Path(relative).name).write_bytes(data)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", str(RUNTIME / "requirements.txt")], check=True)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
sys.path.insert(0, str(RUNTIME))
from manga_tag_policy import Policy
for relative in CONFIG_FILES:
    with urllib.request.urlopen(BASE + relative, timeout=45) as response:
        data = response.read(1_000_001)
    if len(data) > 1_000_000:
        raise RuntimeError("Слишком большой файл правил тегов.")
    Policy(json.loads(data))  # Mutable data only; executable sources remain hash-checked.
    (RUNTIME / Path(relative).name).write_bytes(data)
from run_import import validate_job, run_job
from vault_drive import GoogleDriveAdapter, VaultDrive, VaultError
from source_parser import SourceError
print("Среда готова. Платные ресурсы не включались.")

In [ ]:
# 2. Вставь задание кнопкой «Скопировать задание» на сайте (оно копируется одной строкой).
# Здесь нужны только ссылки и ID каталога. Фразу библиотеки сюда вводить нельзя.
job_text = input("Задание MediaVault (JSON): ").strip()
try:
    job = json.loads(job_text)
    validate_job(job)
except Exception:
    job_text = ""
    raise RuntimeError("Некорректное задание. Скопируйте его кнопкой на сайте заново.") from None
job_text = ""
print("Задание проверено. Далее откроется вход в Google.")

In [ ]:
# 3. Вход владельца, открытие библиотеки и серверный импорт.
from google.colab import auth
from getpass import getpass

auth.authenticate_user()
vault = None
phrase = None
try:
    # Адрес Google-аккаунта проверяется до любых записей в Drive.
    vault = VaultDrive(GoogleDriveAdapter.from_adc())
    phrase = getpass("Секретная фраза открытой библиотеки: ")
    try:
        vault.unlock(job["library_id"], phrase)
    except Exception:
        raise VaultError("Не удалось открыть библиотеку. Проверьте ID, аккаунт и секретную фразу.") from None
    finally:
        phrase = None
    result = run_job(job, vault)
    print("Откройте сайт и нажмите «Обновить библиотеку»:")
    print("https://onikri.github.io/MediaVault/?library=" + vault.library_id)
except (SourceError, VaultError) as error:
    print("Импорт остановлен:", str(error))
    print("Завершённые серии сохранены. После исправления причины можно запустить то же задание.")
except KeyboardInterrupt:
    print("Импорт прерван. Завершённые серии сохранены; то же задание можно продолжить.")
except Exception:
    print("Импорт остановлен из-за ошибки сети, доступа Google или квоты.")
    print("Проверьте свободное место и аккаунт. Повторите запуск; готовые серии будут пропущены.")
finally:
    phrase = None
    if vault is not None:
        vault.close()
    vault = None

### Что сохраняется

Drive содержит каталог, зашифрованные блоки по 4 МиБ и зашифрованные манифесты
восстановления. Разделы называются «Аниме», «Манга», «Ранобэ», «Книги», «Музыка»; имена тайтлов и
видео находятся внутри зашифрованного каталога, а папки тайтлов, сезонов/томов и файлов имеют случайные ID.
Совместимость с веб-приложением проверяется тестами Python ↔ Web Crypto.

Фраза и производный ключ используются в памяти текущего процесса. Авторизацией
Google управляет Colab. Завершите сессию через **Среда выполнения → Отключиться
и удалить среду выполнения**, когда импорт закончен. Во время обычного просмотра
на сайте видео, разумеется, скачивается из Drive на устройство зрителя.

[Исходный код и инструкции](https://github.com/Onikri/MediaVault)
· [Использованные открытые библиотеки](https://github.com/Onikri/MediaVault/blob/main/THIRD_PARTY.md)